In [ ]:
import re
import os
import streamlit as st
import yfinance as yf
import plotly.graph_objects as go
import plotly.express as px
from phi.agent import Agent
from phi.model.groq import Groq
from phi.tools.yfinance import YFinanceTools
from phi.tools.duckduckgo import DDGS
from dotenv import load_dotenv

In [2]:
load_dotenv()

True

In [5]:
def extract_data(ticker, period="6mo"):
    stock = yf.Ticker(ticker)
    hist = stock.history(period=period)    
    hist.reset_index(inplace=True)
    return hist


def plot_stock_price(hist, ticker):
    fig = px.line(hist, x="Date", y="Close", title=f"{ticker} Stock Prices (Last 6 Months)", markers=True)    
    st.plotly_chart(fig)

def plot_candlestick(hist, ticker):
    fig = go.Figure(data=[go.Candlestick(x=hist['Date'], open=hist['Open'], high=hist['High'], low=hist['Low'], close=hist['Close'])])
    fig.update_layout(title=f"{ticker} Candlestick Chart (Last 6 Months)")
    st.plotly_chart(fig)

def plot_moving_average(hist, ticker):
    # Calculates the 20-period Simple Moving Average (SMA) and adds it to the DataFrame.
    hist['SMA_20'] = hist['Close'].rolling(window=20).mean()
    # Calculates the 20-period Exponential Moving Average (EMA) and adds it to the DataFrame.
    hist['EMA_20'] = hist['Close'].ewm(span=20, adjust=False).mean()
    
    fig = px.line(hist, x='Date', y=['Close', 'SMA_20', 'EMA_20'], title=f"{ticker} Moving Average (Last 6 Months)", labels={'value': 'Price (USD)', 'Date': 'Date'})
    st.plotly_chart(fig)

def plot_volume(hist, ticker):
    fig = px.bar(hist, x='Date', y='Volume', title=f"{ticker} Trading Volume (Last 6 Months)")    
    st.plotly_chart(fig)

def search_web(query: str) -> str:
    """Searches DuckDuckGo for the given query and returns results."""
    with DDGS() as ddgs:
        results = [r for r in ddgs.text(query, max_results=5)]
        return str(results)

In [6]:
web_search_agent = Agent(name="Web Search Agent",
                              role="To search the web",
                              model=Groq(id="openai/gpt-oss-120b"),
                              tools=[search_web],
                              instructions=["Always includes the sources"],
                              show_tool_calls=True, markdown=True)

financial_agent = Agent(name="Financial Agent",
                              model=Groq(id="openai/gpt-oss-120b"),
                              tools=[YFinanceTools(stock_price=True,
                                                   analyst_recommendations=True,
                                                   stock_fundamentals=True,
                                                   company_news=True)],
                              instructions=["Use tables to show the data"],
                              show_tool_calls=True, markdown=True)

multi_ai_agent = Agent(team=[web_search_agent, financial_agent],
                       model=Groq(id="llama-3.3-70b-versatile"),
                       instructions=["Always include sources", "Use tables to show the data"],
                       show_tool_calls=True, markdown=True)


In [7]:
ticker = "GOOG"

In [8]:
hist = extract_data(ticker)

In [9]:
ai_response = multi_ai_agent.run(f"Summarize the analyst recomendation and share the last news about {ticker}")

In [10]:
print(ai_response)

content="\nRunning:\n - transfer_task_to_financial_agent(additional_information=Include the number of buy, sell and hold recommendations, expected_output=A summary of the analyst recommendations, task_description=Get the latest analyst recommendations for GOOG)\n - transfer_task_to_financial_agent(additional_information=Include the title and a short description of the news, expected_output=A summary of the latest news, task_description=Get the latest company news for GOOG)\n\n59 analysts (including 14 strong-buy) currently recommend buying GOOG, with no sell or strong-sell recommendations. Here is the latest news from Yahoo Finance and other sources:\n\n*   **Elon Musk wants to put data centers in space — here's what that could actually look like** (May 24, 14:30): Elon Musk proposes locating data-center hardware in orbit to boost efficiency.\n*   **Trump traded over $50 million in ‘Magnificent 7’ stocks last quarter, loading up on Apple and Google and selling Tesla** (May 19, 10:00): 